# Applied AI & Engineering - Notebook 25
## Industrial Model Deployment & Web Applications (Streamlit & Flask)

Welcome to Module 25.

In Notebooks 13 through 24, you learned how to process data, train machine learning models, build deep neural networks, and forecast time series. But as long as your model lives inside a Jupyter Notebook on your personal laptop, it creates zero business value.

An AI model only creates business value when it is **Deployed** into production where end-users, mobile apps, web applications, and enterprise software can query it 24/7.

In this notebook, we master industrial model deployment, REST API engineering, and interactive web application design using an anchor scenario:
* **Deploying an Enterprise Production AI Model & Prediction Portal:** Serializing a trained credit risk evaluation model, building a lightweight production REST API using Flask, designing an interactive executive web portal using Streamlit, and implementing strict API input validation and error handling.

For every topic, we answer three questions:
* What is this in plain English?
* Why are we learning this?
* How is this used in Artificial Intelligence?

---
## Section 1: Model Serialization & Persistence
### Pickle, Joblib, & ONNX Export Pipelines

### 1. What is this in plain English?
Training a Machine Learning or Deep Learning model can take hours, days, or weeks of heavy GPU computation. Once training is complete, you cannot afford to retrain the model every time a user requests a prediction.

**Model Serialization** is the process of taking the trained mathematical brain (weights, biases, tree split thresholds) and freezing it into a single binary file on disk. 

When your production web server starts up, it loads that binary file into memory in milliseconds, ready to serve instant predictions.

### 2. The Serialization Standards:
* **Joblib:** The industry standard for Scikit-Learn models containing large NumPy arrays. Fast, efficient, and lightweight.
* **Pickle:** Python's built-in object serialization library. Flexible, but slower than Joblib for large numeric arrays.
* **ONNX (Open Neural Network Exchange):** An open cross-platform format for running models at maximum speed in C++, Java, or web browsers regardless of whether they were originally trained in PyTorch, TensorFlow, or Scikit-Learn.

### 3. Critical Production Rule: Preprocessor Persistence
If you scaled your features using `StandardScaler` or encoded categories using `OneHotEncoder` during training, you MUST serialize those exact scaler and encoder objects alongside your model! The production API must transform raw incoming user data using the exact same scaling parameters learned during training.

In [1]:
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

# GENERATE ENTERPRISE CREDIT RISK DATASET
np.random.seed(42)
sample_size = 500

income = np.random.normal(65000, 15000, sample_size)
credit_score = np.random.normal(680, 50, sample_size)
debt_ratio = np.random.uniform(0.1, 0.6, sample_size)

# Target: Approved (1) vs Default Risk (0)
score = (income / 15000) + (credit_score / 100) - (debt_ratio * 4)
approved = (score > np.median(score)).astype(int)

df = pd.DataFrame({
    "Income": income,
    "Credit_Score": credit_score,
    "Debt_Ratio": debt_ratio,
    "Approved": approved
})

X = df[["Income", "Credit_Score", "Debt_Ratio"]]
y = df["Approved"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# STEP 1: FIT PREPROCESSOR & MODEL
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

model = RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42)
model.fit(X_train_scaled, y_train)

# STEP 2: SERIALIZE MODEL AND PREPROCESSOR TO DISK USING JOBLIB
joblib.dump(model, "credit_model.joblib")
joblib.dump(scaler, "credit_scaler.joblib")

print("--- Model & Preprocessor Serialization Complete ---")
print("Saved artifacts: 'credit_model.joblib' and 'credit_scaler.joblib'")

# STEP 3: SIMULATE PRODUCTION LOADING AND INFERENCE
# Load frozen artifacts from disk in a fresh session
loaded_model = joblib.load("credit_model.joblib")
loaded_scaler = joblib.load("credit_scaler.joblib")

# Incoming raw customer data from web request: [Income, Credit_Score, Debt_Ratio]
raw_customer = np.array([[75000.0, 710.0, 0.25]])

# Scale using loaded scaler -> Predict using loaded model
scaled_customer = loaded_scaler.transform(raw_customer)
prediction = loaded_model.predict(scaled_customer)[0]
probability = loaded_model.predict_proba(scaled_customer)[0, 1]

print("\n--- Production Inference Results ---")
print(f"Customer Features: Income=$75,000 | Credit Score=710 | Debt Ratio=0.25")
print(f"Approval Prediction:  {'APPROVED (1)' if prediction == 1 else 'DENIED (0)'}")
print(f"Approval Probability: {probability * 100:.2f}%")

--- Model & Preprocessor Serialization Complete ---
Saved artifacts: 'credit_model.joblib' and 'credit_scaler.joblib'

--- Production Inference Results ---
Customer Features: Income=$75,000 | Credit Score=710 | Debt Ratio=0.25
Approval Prediction:  APPROVED (1)
Approval Probability: 89.59%


C:\Users\moham\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


---
## Section 2: REST API Architecture with Flask
### Building Lightweight Prediction Endpoints

### 1. What is this in plain English?
A **REST API (Application Programming Interface)** acts like a digital drive-thru window for software:
* A client (a mobile phone app, a website, or another backend server) sends a JSON data payload over the internet to a specific URL endpoint (e.g., `https://api.company.com/v1/predict`).
* The API server processes the JSON request, transforms the features, passes them into the serialized AI model, and returns a JSON response payload containing the prediction result.

### 2. HTTP Verbs & Status Codes:
* **POST Request:** Used to submit data payloads to the API endpoint for processing.
* **JSON (JavaScript Object Notation):** The universal lightweight text format for sending structured data over the web (e.g., `{"income": 75000, "credit_score": 710}`).
* **200 OK:** Successful request and prediction.
* **400 Bad Request:** User sent invalid, missing, or corrupt input data.
* **500 Internal Server Error:** Server-side code crashed during computation.

### 3. Flask Framework:
Flask is a lightweight Python web framework used to create microservices and prediction endpoints rapidly.

In [15]:
%%writefile app.py
from flask import Flask, request, jsonify
import joblib
import numpy as np

# Initialize Flask Application
app = Flask(__name__)

# Load Serialized Model and Scaler at startup
try:
    model = joblib.load("credit_model.joblib")
    scaler = joblib.load("credit_scaler.joblib")
    model_loaded = True
except Exception as e:
    model_loaded = False

@app.route("/", methods=["GET"])
def health_check():
    """Health check endpoint to verify server is online."""
    return jsonify({
        "status": "online",
        "service": "Credit Risk Prediction API",
        "model_loaded": model_loaded
    }), 200

@app.route("/predict", methods=["POST"])
def predict_credit_risk():
    """Prediction endpoint for evaluating credit risk."""
    if not model_loaded:
        return jsonify({"error": "Model artifacts not found on server."}), 500

    try:
        # Step 1: Parse JSON request body
        data = request.get_json()

        # Step 2: Validate required input fields
        required_fields = ["income", "credit_score", "debt_ratio"]
        for field in required_fields:
            if field not in data:
                return jsonify({"error": f"Missing required input field: '{field}'"}), 400

        # Step 3: Extract and validate data types
        income = float(data["income"])
        credit_score = float(data["credit_score"])
        debt_ratio = float(data["debt_ratio"])

        if income < 0 or credit_score < 300 or credit_score > 850 or debt_ratio < 0:
            return jsonify({"error": "Input numerical values out of logical bounds."}), 400

        # Step 4: Transform features and execute prediction
        input_array = np.array([[income, credit_score, debt_ratio]])
        scaled_input = scaler.transform(input_array)

        prediction = int(model.predict(scaled_input)[0])
        probability = float(model.predict_proba(scaled_input)[0, 1])

        # Step 5: Construct JSON response
        response = {
            "prediction": prediction,
            "status": "APPROVED" if prediction == 1 else "DENIED",
            "approval_probability": round(probability, 4),
            "inputs_received": {
                "income": income,
                "credit_score": credit_score,
                "debt_ratio": debt_ratio
            }
        }
        return jsonify(response), 200

    except ValueError:
        return jsonify({"error": "Invalid data type. Inputs must be numbers."}), 400
    except Exception as e:
        return jsonify({"error": f"Internal server error: {str(e)}"}), 500

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000, debug=False)

Overwriting app.py


In [22]:
import requests

url = "http://127.0.0.1:5000/predict"

payload = {
    "income": 50000,
    "credit_score": 850,
    "debt_ratio": 0.01
}

response = requests.post(url, json=payload)

print("Status Code:", response.status_code)
print("Response:", response.json())

Status Code: 200
Response: {'approval_probability': 0.696, 'inputs_received': {'credit_score': 850.0, 'debt_ratio': 0.01, 'income': 50000.0}, 'prediction': 1, 'status': 'APPROVED'}


In [5]:
# SIMULATING LOCAL API TESTING USING FLASK TEST CLIENT

import json
from app import app as flask_app

# Create a test client for Flask without starting an external server process
test_client = flask_app.test_client()

print("--- 1. Testing Health Check Endpoint (GET /) ---")
response_health = test_client.get("/")
print("HTTP Status Code:", response_health.status_code)
print("Response JSON:   ", response_health.get_json())

print("\n--- 2. Testing Valid Prediction Request (POST /predict) ---")
valid_payload = {
    "income": 85000,
    "credit_score": 740,
    "debt_ratio": 0.20
}
response_valid = test_client.post("/predict", data=json.dumps(valid_payload), content_type="application/json")
print("HTTP Status Code:", response_valid.status_code)
print("Response JSON:   ", response_valid.get_json())

print("\n--- 3. Testing Invalid Request - Missing Field (POST /predict) ---")
invalid_payload = {
    "income": 85000
    # Missing credit_score and debt_ratio
}
response_invalid = test_client.post("/predict", data=json.dumps(invalid_payload), content_type="application/json")
print("HTTP Status Code:", response_invalid.status_code)
print("Response JSON:   ", response_invalid.get_json())

--- 1. Testing Health Check Endpoint (GET /) ---
HTTP Status Code: 200
Response JSON:    {'model_loaded': True, 'service': 'Credit Risk Prediction API', 'status': 'online'}

--- 2. Testing Valid Prediction Request (POST /predict) ---
HTTP Status Code: 200
Response JSON:    {'approval_probability': 0.9799, 'inputs_received': {'credit_score': 740.0, 'debt_ratio': 0.2, 'income': 85000.0}, 'prediction': 1, 'status': 'APPROVED'}

--- 3. Testing Invalid Request - Missing Field (POST /predict) ---
HTTP Status Code: 400
Response JSON:    {'error': "Missing required input field: 'credit_score'"}


C:\Users\moham\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


---
## Section 3: Interactive Web Applications with Streamlit
### Building Production AI Portals

### 1. What is this in plain English?
Flask APIs are built for machines to talk to machines. But what if you need to build a user interface for non-technical stakeholders (executives, loan officers, doctors) to interact with your AI model?

**Streamlit** is a Python framework that converts a pure Python script into an interactive, professional web application in minutes. You do not need to write a single line of HTML, CSS, or JavaScript!

### 2. Streamlit UI Widgets:
* `st.title()` / `st.markdown()`: Text headings and documentation.
* `st.number_input()` / `st.slider()`: Numerical input controls.
* `st.selectbox()`: Dropdown menus for categorical inputs.
* `st.button()`: Action triggers.
* `st.metric()`: Displaying KPI metrics and prediction scores cleanly.
* `st.dataframe()`: Rendering interactive tables.

### 3. How to Run Streamlit:
In your terminal, run: `streamlit run streamlit_app.py`

In [23]:
%%writefile streamlit_app.py
import streamlit as st
import joblib
import numpy as np

# Set page title and layout
st.set_page_config(page_title="Credit Risk AI Portal", layout="centered")

st.title("Enterprise Credit Risk Evaluation Portal")
st.markdown("Use this portal to evaluate applicant creditworthiness in real time using our deployed Random Forest model.")

# Load Model Artifacts
@st.cache_resource
def load_artifacts():
    model = joblib.load("credit_model.joblib")
    scaler = joblib.load("credit_scaler.joblib")
    return model, scaler

try:
    model, scaler = load_artifacts()
    artifacts_ok = True
except Exception as e:
    st.error("Error loading model artifacts. Ensure model files exist.")
    artifacts_ok = False

if artifacts_ok:
    st.subheader("Applicant Details")
    
    # Input Widgets
    income = st.number_input("Annual Income (USD)", min_value=0, max_value=500000, value=65000, step=1000)
    credit_score = st.slider("Credit Score", min_value=300, max_value=850, value=700, step=1)
    debt_ratio = st.slider("Debt-to-Income Ratio", min_value=0.0, max_value=1.0, value=0.30, step=0.01)

    if st.button("Evaluate Credit Risk"):
        # Format and Scale Inputs
        input_data = np.array([[income, credit_score, debt_ratio]])
        scaled_input = scaler.transform(input_data)

        # Run Prediction
        prediction = model.predict(scaled_input)[0]
        probability = model.predict_proba(scaled_input)[0, 1]

        st.markdown("---")
        st.subheader("Evaluation Output")

        col1, col2 = st.columns(2)

        with col1:
            if prediction == 1:
                st.success("Decision: APPROVED")
            else:
                st.error("Decision: DENIED")

        with col2:
            st.metric(label="Approval Probability", value=f"{probability * 100:.1f}%")

        st.progress(float(probability))

Overwriting streamlit_app.py


---
## Section 4: API Input Validation, Security & Defensive Coding

### 1. What is this in plain English?
In production, your API endpoint will be exposed to the public internet. You must write **Defensive Code** to handle malicious or corrupted requests.

Common Threats & Production Safeguards:
1. **Type Mismatch:** A user passes string `"high"` for `income`.
   * *Fix:* Enforce explicit type casting (`float(value)`) wrapped in `try/except ValueError` blocks.
2. **Out-of-Bounds Values:** A user passes `credit_score = 9999`.
   * *Fix:* Implement logical range checks before passing data to the model.
3. **Missing Keys:** A user passes an incomplete JSON payload.
   * *Fix:* Validate required fields against a predefined schema list.
4. **Data Leakage in API:** Scalers must NEVER be re-fitted on incoming API requests (`.fit_transform()` is forbidden in APIs!). Use ONLY `.transform()`.

In [7]:
# DEFENSIVE INPUT VALIDATION PIPELINE FUNCTION

def validate_and_process_payload(json_payload, required_schema):
    """
    Validates an incoming JSON payload against a required schema and logical bounds.
    """
    # Check for missing keys
    for field, field_type in required_schema.items():
        if field not in json_payload:
            raise KeyError(f"Missing required payload key: '{field}'")
        
        # Check for type validity
        try:
            val = field_type(json_payload[field])
        except (ValueError, TypeError):
            raise TypeError(f"Field '{field}' must be convertible to {field_type.__name__}")

    # Logical domain bounds
    income = float(json_payload["income"])
    score = float(json_payload["credit_score"])
    
    if income < 0 or score < 300 or score > 850:
        raise ValueError("Field values out of acceptable operational bounds.")
        
    return True

# Test Defensive Validation Pipeline
schema = {"income": float, "credit_score": float, "debt_ratio": float}

# Test Case 1: Valid
try:
    validate_and_process_payload({"income": 50000, "credit_score": 680, "debt_ratio": 0.3}, schema)
    print("Validation Test 1 (Valid Payload): PASSED")
except Exception as e:
    print("Validation Test 1 Failed:", e)

# Test Case 2: Out of bounds credit score
try:
    validate_and_process_payload({"income": 50000, "credit_score": 950, "debt_ratio": 0.3}, schema)
    print("Validation Test 2 Failed: Should have rejected invalid score.")
except Exception as e:
    print("Validation Test 2 (Out of Bounds Rejected): PASSED | Error Caught:", e)

Validation Test 1 (Valid Payload): PASSED
Validation Test 2 (Out of Bounds Rejected): PASSED | Error Caught: Field values out of acceptable operational bounds.


---
## Hands-on Workshop: Model Deployment & Web Applications

Complete these four practical challenges to build your deployment engineering capabilities.

### Challenge 1: Model & Scaler Serialization (Junior Level)

**Scenario:** Train a Logistic Regression model on a synthetic dataset and save both the model and scaler using Joblib.

**Task Requirements:**
1. Given `X_train` and `y_train` below.
2. Fit `StandardScaler` and `LogisticRegression()`.
3. Save scaler to `'challenge_scaler.joblib'` and model to `'challenge_model.joblib'`.
4. Load both back into memory and verify they exist.

In [9]:
# --- CHALLENGE 1: YOUR WORKSPACE ---


# --- CHALLENGE 1: INSTRUCTOR SOLUTION ---
import numpy as np
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

np.random.seed(42)
X_train = np.random.normal(50, 10, (100, 2))
y_train = (X_train[:, 0] + X_train[:, 1] > 100).astype(int)

# Fit
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_train)

model = LogisticRegression()
model.fit(X_scaled, y_train)

# Save
joblib.dump(scaler, "challenge_scaler.joblib")
joblib.dump(model, "challenge_model.joblib")

# Load
loaded_s = joblib.load("challenge_scaler.joblib")
loaded_m = joblib.load("challenge_model.joblib")

print("Challenge 1 Serialization Check: Artifacts saved and loaded successfully.")

Challenge 1 Serialization Check: Artifacts saved and loaded successfully.


### Challenge 2: JSON Payload Schema Validator (Mid Level)

**Scenario:** Build an API payload validator for a housing price prediction service.

**Task Requirements:**
1. Required fields: `["sqft", "bedrooms", "age"]`.
2. Write a function `validate_housing_payload(payload)`.
3. Return `False, "Error message"` if any field is missing or if `sqft <= 0`.
4. Return `True, "Valid"` if input passes all checks.

In [10]:
# --- CHALLENGE 2: YOUR WORKSPACE ---


# --- CHALLENGE 2: INSTRUCTOR SOLUTION ---
def validate_housing_payload(payload):
    required = ["sqft", "bedrooms", "age"]
    for req in required:
        if req not in payload:
            return False, f"Missing required field: {req}"
            
    try:
        sqft = float(payload["sqft"])
        if sqft <= 0:
            return False, "Square footage must be greater than zero"
    except (ValueError, TypeError):
        return False, "Square footage must be a valid number"
        
    return True, "Valid"

# Test
print(validate_housing_payload({"sqft": 1500, "bedrooms": 3, "age": 10}))
print(validate_housing_payload({"bedrooms": 3, "age": 10}))
print(validate_housing_payload({"sqft": -500, "bedrooms": 3, "age": 10}))

(True, 'Valid')
(False, 'Missing required field: sqft')
(False, 'Square footage must be greater than zero')


### Challenge 3: Flask Endpoint Construction (Senior Level)

**Scenario:** Construct a complete Flask POST prediction endpoint for a sales prediction service.

**Task Requirements:**
1. Create a Flask app `app_sales = Flask(__name__)`.
2. Define route `/predict_sales` accepting `POST`.
3. Extract `ad_spend` from JSON payload.
4. Calculate predicted sales: `sales = ad_spend * 2.5`.
5. Return JSON response `{"predicted_sales": sales}` with HTTP status 200.

In [12]:
# --- CHALLENGE 3: YOUR WORKSPACE ---


# --- CHALLENGE 3: INSTRUCTOR SOLUTION ---
from flask import Flask, request, jsonify

app_sales = Flask("sales_app")

@app_sales.route("/predict_sales", methods=["POST"])
def predict_sales():
    data = request.get_json()
    if not data or "ad_spend" not in data:
        return jsonify({"error": "Missing ad_spend"}), 400
        
    ad_spend = float(data["ad_spend"])
    predicted_sales = ad_spend * 2.5
    return jsonify({"predicted_sales": predicted_sales}), 200

# Test using test client
client = app_sales.test_client()
res = client.post("/predict_sales", json={"ad_spend": 1000})
print("Challenge 3 Flask API Output:", res.get_json())

Challenge 3 Flask API Output: {'predicted_sales': 2500.0}


### Challenge 4: Streamlit UI File Generator (Architect Level)

**Scenario:** Write a Streamlit application script that takes house features and displays a formatted prediction metric.

**Task Requirements:**
1. Use `%%writefile challenge_streamlit.py`.
2. Import `streamlit as st`.
3. Add a title `"Real Estate Price Predictor"`.
4. Add a number input for `sqft` (min 500, max 10000, value 2000).
5. Add a button `"Calculate Estimate"`. When clicked, display `st.metric("Estimated Price", f"${sqft * 200:,}")`.

In [13]:
# --- CHALLENGE 4: YOUR WORKSPACE ---


# --- CHALLENGE 4: INSTRUCTOR SOLUTION ---
%%writefile challenge_streamlit.py
import streamlit as st

st.title("Real Estate Price Predictor")
st.write("Enter house features below to generate an automated price estimate.")

sqft = st.number_input("Square Footage", min_value=500, max_value=10000, value=2000, step=100)

if st.button("Calculate Estimate"):
    estimated_price = sqft * 200
    st.metric(label="Estimated Price", value=f"${estimated_price:,}")

UsageError: Line magic function `%%writefile` not found.


# Session Summary

Congratulations. You have completed Notebook 25.

Let us review what you now understand:
1. **Model Serialization:** You know how to freeze trained models and scalers to disk using `joblib` and load them instantly in production environments.
2. **REST API Architecture:** You understand how Flask uses HTTP POST requests and JSON payloads to serve prediction microservices.
3. **Streamlit Applications:** You know how to turn Python scripts into interactive, web-based AI portals with sliders, buttons, and metrics.
4. **Defensive API Coding:** You know how to validate JSON schemas, handle data type errors, and return proper HTTP status codes (`200 OK`, `400 Bad Request`).